In [2]:
!pip install pyspark findspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inicializar SparkSession
spark = SparkSession.builder \
    .appName("Guia03_Exploracion_Spark") \
    .getOrCreate()

ruta_archivo = "../datos/procesados/dataset_limpio_guia02.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(ruta_archivo)

print("¡Dataset cargado exitosamente!")

¡Dataset cargado exitosamente!


In [4]:
# 2 Visualizar el esquema del DataFrame
df.printSchema()

root
 |-- registro_id: integer (nullable = true)
 |-- empleado_id: string (nullable = true)
 |-- periodo: string (nullable = true)
 |-- fecha_evaluacion: date (nullable = true)
 |-- productividad_pct: double (nullable = true)
 |-- calidad_trabajo_pct: double (nullable = true)
 |-- cumplimiento_objetivos_pct: double (nullable = true)
 |-- asistencia_pct: double (nullable = true)
 |-- horas_capacitacion: double (nullable = true)
 |-- horas_extra: integer (nullable = true)
 |-- ausencias_dias: double (nullable = true)
 |-- puntaje_desempeno: double (nullable = true)
 |-- nivel_desempeno: string (nullable = true)
 |-- sede_codigo: string (nullable = true)
 |-- nombre_sede: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- departamento_geo: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- direccion_referencia: string (nullable = true)
 |-- latitud: double (nullable = true)
 |-- longitud: double (nullable = true)
 

In [5]:
# 2.2 
columnas_clave = ["empleado_id", "nombre_completo", "departamento", "salario_mensual", "puntaje_desempeno"]
df.select(columnas_clave).show(5, truncate=False)

# 2.3 
df.select("salario_mensual", "puntaje_desempeno", "productividad_pct", "horas_capacitacion").describe().show()

+-----------+---------------+------------+---------------+-----------------+
|empleado_id|nombre_completo|departamento|salario_mensual|puntaje_desempeno|
+-----------+---------------+------------+---------------+-----------------+
|EMP001     |Ana Hernández  |Tecnología  |1400.0         |77.55            |
|EMP001     |Ana Hernández  |Tecnología  |1400.0         |72.73            |
|EMP001     |Ana Hernández  |Tecnología  |1400.0         |77.06            |
|EMP001     |Ana Hernández  |Tecnología  |1400.0         |79.61            |
|EMP002     |Carlos Vásquez |Tecnología  |1450.0         |84.31            |
+-----------+---------------+------------+---------------+-----------------+
only showing top 5 rows
+-------+------------------+------------------+-----------------+------------------+
|summary|   salario_mensual| puntaje_desempeno|productividad_pct|horas_capacitacion|
+-------+------------------+------------------+-----------------+------------------+
|  count|               168|

In [6]:
# Transformación 1: filter()
df_filtrado = df.filter((F.col("estado_laboral") == "Activo") & (F.col("puntaje_desempeno") >= 85))
df_filtrado.select("empleado_id", "nombre_completo", "departamento", "puntaje_desempeno").show(5)

# Transformación 2: groupBy()
df_agrupado = df.groupBy("departamento").agg(
    F.count("registro_id").alias("total_evaluaciones"),
    F.round(F.avg("salario_mensual"), 2).alias("salario_promedio"),
    F.round(F.avg("puntaje_desempeno"), 2).alias("desempeno_promedio")
)
df_agrupado.show()

# Transformación 3: orderBy()
df_ordenado = df_agrupado.orderBy(F.col("desempeno_promedio").desc())
df_ordenado.show()

+-----------+---------------+------------+-----------------+
|empleado_id|nombre_completo|departamento|puntaje_desempeno|
+-----------+---------------+------------+-----------------+
|     EMP007|  Daniela López| Operaciones|            87.46|
|     EMP007|  Daniela López| Operaciones|             85.0|
|     EMP007|  Daniela López| Operaciones|            90.33|
|     EMP008| Miguel Ramírez| Operaciones|            94.16|
|     EMP008| Miguel Ramírez| Operaciones|            95.47|
+-----------+---------------+------------+-----------------+
only showing top 5 rows
+----------------+------------------+----------------+------------------+
|    departamento|total_evaluaciones|salario_promedio|desempeno_promedio|
+----------------+------------------+----------------+------------------+
|Recursos Humanos|                32|          1412.5|             80.73|
|        Finanzas|                34|         1216.18|             83.88|
|     Operaciones|                34|         1269.12|   

In [7]:
# 4.1 Vista temporal
df.createOrReplaceTempView("evaluaciones_personal")

# 4.2 Ejecutar consulta SQL
query_sql = """
    SELECT 
        departamento,
        modalidad,
        COUNT(empleado_id) AS total_colaboradores,
        ROUND(AVG(salario_mensual), 2) AS salario_medio,
        ROUND(AVG(puntaje_desempeno), 2) AS desempeno_medio
    FROM evaluaciones_personal
    WHERE estado_laboral = 'Activo'
    GROUP BY departamento, modalidad
    ORDER BY departamento ASC, desempeno_medio DESC
"""

df_sql_resultado = spark.sql(query_sql)
df_sql_resultado.show(truncate=False)

+----------------+----------+-------------------+-------------+---------------+
|departamento    |modalidad |total_colaboradores|salario_medio|desempeno_medio|
+----------------+----------+-------------------+-------------+---------------+
|Comercial       |Híbrida   |4                  |1700.0       |89.95          |
|Comercial       |Remota    |8                  |1275.0       |88.84          |
|Comercial       |Presencial|12                 |1516.67      |83.02          |
|Finanzas        |Presencial|9                  |1188.89      |87.43          |
|Finanzas        |Remota    |12                 |1233.33      |84.44          |
|Finanzas        |Híbrida   |8                  |1200.0       |83.1           |
|Operaciones     |Remota    |4                  |1450.0       |86.7           |
|Operaciones     |Presencial|4                  |850.0        |85.73          |
|Operaciones     |Híbrida   |22                 |1234.09      |85.53          |
|Recursos Humanos|Híbrida   |4          

In [8]:
# Detener la sesión de Spark liberar memoria
spark.stop()